# FireBridge 4/5: Harmonization (version 2)
**Question:** how many VIIRS detections would each 10 km grid cell have had in each month, given everything MODIS saw there and around it?

Answering it for 2000–2012 rebuilds the pre-VIIRS record on the VIIRS scale: one consistent burning-activity calendar per region, 2000–2024.

**Why cell × month instead of per MODIS detection (notebook 3):**
- *Blind spot:* most VIIRS fires happen where no MODIS pixel saw anything. Here every cell that ever burned is a row, including months where MODIS saw nothing, so the model learns how much MODIS misses.
- *Terra:* both MODIS satellites are inputs, so 2000–2002 (Terra only) is covered.
- *What the calendar needs:* totals per area and month.

**Input:** `FireBridge/data/processed/<area>_clean.parquet` (notebook 2) and `data/regions.csv` (notebook 1).  
**Output:** `FireBridge/data/harmonized/harmonized_cells.parquet`, `harmonized_monthly.parquet`, model in `FireBridge/models/`.

CPU is fine. XGBoost uses the GPU automatically on a GPU runtime.

In [ ]:
!pip -q install duckdb xgboost
from google.colab import drive
drive.mount('/content/drive')

## Functions
Run once per session. Settings at the top: `GRID_KM` (10), `START`/`END` (2000–2024), `VIIRS_FROM` (2013), the year split, and `SEEDS` for the ensemble.

Data: `coverage`, `buildCellMonth`, `addRegionColumns`, `splitByYear`.  
Models: `RatioBaseline` (per-region VIIRS/MODIS ratio), `GBMPoisson` (plain gradient boosting, Poisson loss), `OffsetGBM` (**main**: the ratio × a learned correction).  
Evaluation: `compareModels`, `finalTest`, `leaveOneRegionOut`.  
Results: `trainEnsemble`, `harmonize`, `jumpCheck`, `plotYearly`, `plotCalendar`, `anomalies`, `saveOutputs`.

In [ ]:
import os, glob, ast, json, math
import numpy as np
import pandas as pd
import duckdb
import joblib
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_poisson_deviance

DATA_DIR = '/content/drive/MyDrive/FireBridge/data'
PROCESSED_DIR = f'{DATA_DIR}/processed'          # <area>_clean.parquet from notebook 2
OUTPUT_DIR = f'{DATA_DIR}/harmonized'            # results of this notebook
MODEL_DIR = '/content/drive/MyDrive/FireBridge/models'

GRID_KM = 10              # grid cell size (km) on the EPSG:6933 equal-area grid
START, END = 2000, 2024   # years covered by the calendar
VIIRS_FROM = 2013         # first full VIIRS year used as a target (the plan's 2013-2024 range)
TRAIN_UNTIL, VAL_YEARS, TEST_FROM = 2020, (2021,), 2022
SEEDS = (0, 1, 2, 3, 4)   # ensemble members -> uncertainty band
RANDOM_STATE = 42

# EPSG:6933 (EASE-Grid 2.0 global, equal area) written out, so no pyproj is needed
_R, _K0 = 6371007.181, math.cos(math.radians(30))
_X = f"{_R * _K0} * radians(longitude)"
_Y = f"{_R / _K0} * sin(radians(latitude))"

BASE_FEATURES = ['modis_count', 'terra_count', 'aqua_count', 'frp_sum', 'frp_max', 'frp_mean',
                 'pixel_area_mean', 'day_share', 'nb_count', 'cell_mean', 'cell_month_mean',
                 'month_sin', 'month_cos']
TARGET = 'viirs_count'


# ======================= regions and data =======================
def loadRegions():
    """Region table written by notebook 1 (bbox, focus period, fire type). Empty if not found."""
    p = f'{DATA_DIR}/regions.csv'
    if not os.path.exists(p):
        print('regions.csv not found; event checks will be skipped.')
        return pd.DataFrame(columns=['area'])
    r = pd.read_csv(p)
    r['bbox'] = r['bbox'].apply(ast.literal_eval)
    return r


def availableAreas():
    """Region names as they appear in the data (e.g. 'Egypt_Nile_Delta'), one per preprocessed file."""
    names = []
    for f in sorted(glob.glob(f'{PROCESSED_DIR}/*_clean.parquet')):
        names += [r[0] for r in duckdb.sql(f"SELECT DISTINCT country FROM read_parquet('{f}')").fetchall()]
    return sorted(set(names))


def _files(areas):
    files = [f'{PROCESSED_DIR}/{a.lower()}_clean.parquet' for a in areas]
    missing = [f for f in files if not os.path.exists(f)]
    if missing:
        raise FileNotFoundError(f'Not preprocessed yet: {missing}. Run notebook 2 for these areas.')
    return files


def coverage(areas=None):
    """Years available per area and sensor in the preprocessed files. The backfill needs MODIS from 2000."""
    areas = areas or availableAreas()
    t = duckdb.sql(f"""SELECT country AS area, sensor, MIN(year) AS first_year, MAX(year) AS last_year,
                              COUNT(DISTINCT year) AS years, COUNT(*) AS detections
                       FROM read_parquet({_files(areas)}) GROUP BY ALL ORDER BY area, sensor""").df()
    short = t[(t.sensor == 'modis') & (t.first_year > START)]
    if len(short):
        print('MODIS before', short.first_year.min(), 'is missing for:', ', '.join(short.area),
              '\n→ in notebook 1 run downloadRegions(years=range(2000, 2013), sensors=("modis",)), then notebook 2 again.')
    return t


def buildCellMonth(areas, grid_km=GRID_KM, start=START, end=END, viirs_from=VIIRS_FROM):
    """One row per (area, grid cell, month) for every cell that ever had a detection, every month start-end.

    Features come from MODIS only (Terra + Aqua). Target viirs_count is set from `viirs_from` on and
    left empty (NaN) before, where VIIRS did not exist: those are the months the model fills in.
    """
    fires = duckdb.sql(f"SELECT * FROM read_parquet({_files(areas)})")
    g = grid_km * 1000
    df = duckdb.sql(f"""
        WITH f AS (
            SELECT country AS area, sensor, satellite, daynight, frp, scan * track AS pix, year,
                   CAST(floor({_X} / {g}) AS INTEGER) AS gx, CAST(floor({_Y} / {g}) AS INTEGER) AS gy,
                   CAST(date_trunc('month', datetime) AS DATE) AS month
            FROM fires WHERE year BETWEEN {start} AND {end}),
        m AS (
            SELECT area, gx, gy, month, COUNT(*) AS modis_count,
                   SUM(CASE WHEN satellite = 'Terra' THEN 1 ELSE 0 END) AS terra_count,
                   SUM(CASE WHEN satellite = 'Aqua' THEN 1 ELSE 0 END) AS aqua_count,
                   SUM(frp) AS frp_sum, MAX(frp) AS frp_max, AVG(pix) AS pixel_area_mean,
                   AVG(CASE WHEN daynight = 'D' THEN 1.0 ELSE 0.0 END) AS day_share
            FROM f WHERE sensor = 'modis' GROUP BY ALL),
        v AS (
            SELECT area, gx, gy, month, COUNT(*) AS viirs_count, SUM(frp) AS viirs_frp_sum
            FROM f WHERE sensor = 'viirs' AND year >= {viirs_from} GROUP BY ALL),
        cells AS (SELECT DISTINCT area, gx, gy FROM m UNION SELECT DISTINCT area, gx, gy FROM v),
        months AS (SELECT CAST(generate_series AS DATE) AS month
                   FROM generate_series(DATE '{start}-01-01', DATE '{end}-12-01', INTERVAL 1 MONTH)),
        grid AS (SELECT * FROM cells CROSS JOIN months),
        nb AS (    -- MODIS detections in the 8 neighbouring cells
            SELECT m.area, m.gx + dx AS gx, m.gy + dy AS gy, m.month, SUM(m.modis_count) AS nb_count
            FROM m, range(-1, 2) a(dx), range(-1, 2) b(dy)
            WHERE NOT (dx = 0 AND dy = 0) GROUP BY ALL)
        SELECT grid.area, grid.gx, grid.gy, grid.month,
               COALESCE(m.modis_count, 0) AS modis_count, COALESCE(m.terra_count, 0) AS terra_count,
               COALESCE(m.aqua_count, 0) AS aqua_count, COALESCE(m.frp_sum, 0) AS frp_sum,
               COALESCE(m.frp_max, 0) AS frp_max, m.frp_sum / m.modis_count AS frp_mean,
               m.pixel_area_mean, m.day_share, COALESCE(nb.nb_count, 0) AS nb_count,
               CASE WHEN year(grid.month) >= {viirs_from} THEN COALESCE(v.viirs_count, 0) END AS viirs_count,
               CASE WHEN year(grid.month) >= {viirs_from} THEN COALESCE(v.viirs_frp_sum, 0) END AS viirs_frp_sum
        FROM grid
        LEFT JOIN m  USING (area, gx, gy, month)
        LEFT JOIN v  USING (area, gx, gy, month)
        LEFT JOIN nb USING (area, gx, gy, month)""").df()
    df = df.sort_values(['area', 'gx', 'gy', 'month']).reset_index(drop=True)   # fixed order -> reproducible models
    df['month'] = pd.to_datetime(df['month'])
    for c in df.columns:                          # DuckDB returns nullable ints; use plain floats/ints
        if str(df[c].dtype) in ('Int64', 'Int32', 'Float64'):
            df[c] = df[c].astype('float64')
    df['year'] = df['month'].dt.year.astype('int16')
    mo = df['month'].dt.month
    df['month_sin'], df['month_cos'] = np.sin(2 * np.pi * mo / 12), np.cos(2 * np.pi * mo / 12)
    # Long-term MODIS activity of each cell (MODIS only, all years): marks crop-burning cells vs quiet ones
    df['cell_mean'] = df.groupby(['area', 'gx', 'gy'])['modis_count'].transform('mean')
    df['cell_month_mean'] = df.groupby(['area', 'gx', 'gy', mo])['modis_count'].transform('mean')
    for c in df.select_dtypes('float64').columns:
        df[c] = df[c].astype('float32')
    n_cells = df.groupby('area')[['gx', 'gy']].apply(lambda d: len(d.drop_duplicates()))
    print(f'{len(df):,} cell-months · cells per area: {n_cells.to_dict()} · '
          f'{(df["modis_count"] > 0).mean():.1%} of cell-months have MODIS fire')
    return df


def addRegionColumns(df, areas):
    """One-hot region columns (fire type). Left out when testing on an unseen region."""
    for a in areas:
        df[f'region_{a}'] = (df['area'].str.lower() == a.lower()).astype('int8')
    empty = [a for a in areas if df[f'region_{a}'].sum() == 0]
    if empty:
        raise ValueError(f'No rows for {empty}: check the names against df.area.unique()')
    return [f'region_{a}' for a in areas]


def splitByYear(df):
    obs = df[df[TARGET].notna()]
    train = obs[obs.year <= TRAIN_UNTIL]
    val = obs[obs.year.isin(VAL_YEARS)]
    test = obs[obs.year >= TEST_FROM]
    for n, d in [('train', train), ('val', val), ('test', test)]:
        print(f'{n:5s}: {len(d):8,} cell-months  {d.year.min()}–{d.year.max()}  '
              f'VIIRS {int(d[TARGET].sum()):,} · MODIS {int(d.modis_count.sum()):,}')
    return train, val, test


# ======================= models =======================
class RatioBaseline:
    """Per region: total VIIRS / total MODIS over the training years, times each cell's MODIS count.
    Cells where MODIS saw nothing stay 0, so this cannot recover the blind spot."""
    name = 'Average ratio (per region)'
    def fit(self, train, features=None):
        t = train.groupby('area')[[TARGET, 'modis_count']].sum()
        self.ratio_ = (t[TARGET] / t['modis_count'].clip(lower=1)).to_dict()
        self.global_ = train[TARGET].sum() / max(train['modis_count'].sum(), 1)
        return self
    def predict(self, df):
        return df['modis_count'].to_numpy() * df['area'].map(self.ratio_).fillna(self.global_).to_numpy()


class GBMPoisson:
    """Gradient boosting with a Poisson loss on the cell-month features.
    Uses XGBoost (GPU if the runtime has one) when installed, else scikit-learn's HistGradientBoosting."""
    def __init__(self, features, seed=RANDOM_STATE, engine='auto', n_estimators=800):
        self.features, self.seed, self.engine, self.n_estimators = features, seed, engine, n_estimators
        self.name = 'Gradient boosting (Poisson)'
    def fit(self, train, val=None):
        X, y = train[self.features], train[TARGET]
        use_xgb = self.engine in ('auto', 'xgboost')
        if use_xgb:
            try:
                import xgboost as xgb
                device = 'cuda' if _has_gpu() else 'cpu'
                self.model_ = xgb.XGBRegressor(
                    objective='count:poisson', n_estimators=self.n_estimators, learning_rate=0.05, max_depth=6,
                    min_child_weight=5, subsample=0.8, colsample_bytree=0.8, reg_lambda=2.0, tree_method='hist',
                    device=device, random_state=self.seed, early_stopping_rounds=50 if val is not None else None)
                kw = {'eval_set': [(val[self.features], val[TARGET])], 'verbose': False} if val is not None else {}
                self.model_.fit(X, y, **kw)
                self.engine_ = f'xgboost ({device})'
                return self
            except ImportError:
                if self.engine == 'xgboost': raise
        self.model_ = HistGradientBoostingRegressor(loss='poisson', max_iter=self.n_estimators, learning_rate=0.05,
                                                    max_leaf_nodes=31, min_samples_leaf=40, l2_regularization=1.0,
                                                    early_stopping=True, validation_fraction=0.1, random_state=self.seed)
        self.model_.fit(X, y)
        self.engine_ = 'sklearn HistGB'
        return self
    def predict(self, df):
        return np.clip(self.model_.predict(df[self.features]), 0, None)


class OffsetGBM:
    """The main model: the ratio baseline × a learned correction.

    XGBoost (Poisson loss) starts from log(baseline) for every cell-month and learns how much to scale it up
    or down from the MODIS features. Big fires keep scaling with MODIS (plain trees can't predict beyond what
    they saw in training), and cells where MODIS saw nothing start from that region's average VIIRS count
    in such cells, so the blind spot is learned.
    """
    name = 'Ratio × learned correction (main)'
    def __init__(self, features, seed=RANDOM_STATE, n_estimators=800):
        self.features, self.seed, self.n_estimators = features, seed, n_estimators
    def _base(self, df):
        r = df['area'].map(self.ratio_).fillna(self.ratio_all_).to_numpy()
        z = df['area'].map(self.zero_).fillna(self.zero_all_).to_numpy()
        m = df['modis_count'].to_numpy()
        return np.log(m * r + np.where(m == 0, z, 0) + 1e-3)
    def fit(self, train, val=None):
        import xgboost as xgb
        t = train.groupby('area')[[TARGET, 'modis_count']].sum()
        self.ratio_ = (t[TARGET] / t['modis_count'].clip(lower=1)).to_dict()
        self.ratio_all_ = train[TARGET].sum() / max(train['modis_count'].sum(), 1)
        zero = train[train['modis_count'] == 0]
        self.zero_ = zero.groupby('area')[TARGET].mean().to_dict()
        self.zero_all_ = zero[TARGET].mean()
        device = 'cuda' if _has_gpu() else 'cpu'
        self.model_ = xgb.XGBRegressor(
            objective='count:poisson', n_estimators=self.n_estimators, learning_rate=0.05, max_depth=6,
            min_child_weight=5, subsample=0.8, colsample_bytree=0.8, reg_lambda=2.0, tree_method='hist',
            device=device, random_state=self.seed, early_stopping_rounds=50 if val is not None else None)
        kw = {}
        if val is not None:
            kw = {'eval_set': [(val[self.features], val[TARGET])], 'base_margin_eval_set': [self._base(val)], 'verbose': False}
        self.model_.fit(train[self.features], train[TARGET], base_margin=self._base(train), **kw)
        self.engine_ = f'xgboost ({device})'
        return self
    def predict(self, df):
        return self.model_.predict(df[self.features], base_margin=self._base(df))


def _has_gpu():
    try:
        import torch
        return torch.cuda.is_available()
    except ImportError:
        return os.path.exists('/proc/driver/nvidia/version')


# ======================= evaluation =======================
def regionMonthly(df, pred):
    t = df[['area', 'month']].copy()
    t['actual'], t['pred'] = df[TARGET].to_numpy(), pred
    return t.groupby(['area', 'month'])[['actual', 'pred']].sum().reset_index()


def scores(df, pred):
    """Cell-level Poisson deviance + what the calendar shows: monthly totals per region."""
    m = regionMonthly(df, pred)
    out = {'Poisson deviance (cell)': mean_poisson_deviance(df[TARGET], np.clip(pred, 1e-6, None)),
           'Total bias %': 100 * (m.pred.sum() / max(m.actual.sum(), 1e-9) - 1),
           'Monthly WAPE %': 100 * (m.actual - m.pred).abs().sum() / max(m.actual.sum(), 1e-9)}
    out['Monthly corr'] = float(np.corrcoef(m.actual, m.pred)[0, 1]) if m.actual.std() > 0 else np.nan
    return out


def scoresByRegion(df, pred):
    rows = []
    for a in sorted(df.area.unique()):
        k = (df.area == a).to_numpy()
        rows.append({'area': a, **scores(df[k], pred[k])})
    return pd.DataFrame(rows)


def compareModels(train, val, features):
    models = [RatioBaseline(), GBMPoisson(features), OffsetGBM(features)]
    rows, fitted = [], {}
    for m in models:
        m.fit(train) if isinstance(m, RatioBaseline) else m.fit(train, val)
        rows.append({'model': m.name, **scores(val, m.predict(val))})
        fitted[m.name] = m
    return pd.DataFrame(rows), fitted


def finalTest(train, val, test, features):
    """Refit on train + validation, score once on the test years, overall and per region (main model)."""
    trval = pd.concat([train, val])
    models = [RatioBaseline().fit(trval), GBMPoisson(features).fit(trval), OffsetGBM(features).fit(trval)]
    preds = {m.name: m.predict(test) for m in models}
    overall = pd.DataFrame([{'model': n, **scores(test, p)} for n, p in preds.items()])
    main = models[-1]
    per_region = scoresByRegion(test, preds[main.name])
    _plotTestMonthly(test, {models[0].name: preds[models[0].name], main.name: preds[main.name]})
    return main, overall, per_region


def _plotTestMonthly(test, preds):
    areas = sorted(test.area.unique())
    fig, axes = plt.subplots(len(areas), 1, figsize=(13, 3.2 * len(areas)), squeeze=False)
    for ax, a in zip(axes[:, 0], areas):
        k = (test.area == a).to_numpy()
        for i, (name, p) in enumerate(preds.items()):
            m = regionMonthly(test[k], p[k]).set_index('month')
            if i == 0:
                ax.plot(m.index, m.actual, color='black', lw=2, label='VIIRS (actual)')
            ax.plot(m.index, m.pred, lw=1.4, label=name)
        ax.set_title(f'{a}: monthly VIIRS detections, test years'); ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()


def leaveOneRegionOut(df_obs, features_no_region):
    """Train on all other regions, test on the held-out one (2013-2024). Region columns are not used."""
    rows = []
    for a in sorted(df_obs.area.unique()):
        tr, te = df_obs[df_obs.area != a], df_obs[df_obs.area == a]
        if tr.empty:
            print('Need at least 2 regions.'); return pd.DataFrame()
        base = RatioBaseline().fit(tr)
        main = OffsetGBM(features_no_region).fit(tr)
        rows.append({'held_out': a, 'model': 'Average ratio (other regions)', **scores(te, base.predict(te))})
        rows.append({'held_out': a, 'model': 'Ratio × learned correction', **scores(te, main.predict(te))})
    return pd.DataFrame(rows)


# ======================= ensemble, backfill, outputs =======================
def trainEnsemble(df_obs, features, seeds=SEEDS):
    """Final models on ALL observed years (2013-2024), one per seed. Their spread = the uncertainty band."""
    members = []
    for s in seeds:
        boot = df_obs.sample(frac=1.0, replace=True, random_state=s)   # bootstrap resample of cell-months
        members.append(OffsetGBM(features, seed=s).fit(boot))
    print(f'{len(members)} members trained ({members[0].engine_}).')
    return members


def harmonize(df, members):
    """Predicted VIIRS-equivalent detections for every cell-month 2000-2024, with an ensemble band."""
    P = np.column_stack([m.predict(df) for m in members])
    out = df[['area', 'gx', 'gy', 'month', 'year', 'modis_count', 'terra_count', 'aqua_count', TARGET]].copy()
    out['harmonized'] = P.mean(axis=1)
    out['low'], out['high'] = np.percentile(P, 10, axis=1), np.percentile(P, 90, axis=1)
    # Region-month totals: sum each member first, then take the spread (cells are not independent)
    t = df[['area', 'month']].copy()
    for i in range(P.shape[1]):
        t[f'm{i}'] = P[:, i]
    tot = t.groupby(['area', 'month']).sum()
    monthly = out.groupby(['area', 'month'])[['modis_count', 'terra_count', 'aqua_count']].sum()
    monthly['viirs'] = out.groupby(['area', 'month'])[TARGET].sum(min_count=1)
    monthly['harmonized'] = tot.mean(axis=1)
    monthly['low'], monthly['high'] = tot.quantile(0.1, axis=1), tot.quantile(0.9, axis=1)
    # The calendar shows real VIIRS where it exists and the model before that
    monthly['calendar'] = monthly['viirs'].fillna(monthly['harmonized'])
    monthly['source'] = np.where(monthly['viirs'].notna(), 'VIIRS', 'model')
    return out, monthly.reset_index()


def jumpCheck(monthly):
    """How big is the step at the sensor switch? Compares 2009-2011 with 2013-2015.

    raw jump         : VIIRS 2013-15 / MODIS 2009-11 (what you get by joining the two records)
    real change      : MODIS 2013-15 / MODIS 2009-11 (one sensor across the switch = true activity change)
    harmonized jump  : VIIRS 2013-15 / harmonized 2009-11 (should be close to the real change)
    """
    rows = []
    for a, d in monthly.groupby('area'):
        y = d.groupby(d.month.dt.year)[['modis_count', 'viirs', 'harmonized']].sum(min_count=1)
        before, after = y.loc[2009:2011].mean(), y.loc[2013:2015].mean()
        rows.append({'area': a,
                     'raw jump ×': after['viirs'] / max(before['modis_count'], 1e-9),
                     'real change ×': after['modis_count'] / max(before['modis_count'], 1e-9),
                     'harmonized jump ×': after['viirs'] / max(before['harmonized'], 1e-9)})
    return pd.DataFrame(rows).round(2)


def plotYearly(monthly):
    for a, d in monthly.groupby('area'):
        y = d.groupby(d.month.dt.year)[['modis_count', 'viirs', 'harmonized', 'low', 'high']].sum(min_count=1)
        fig, ax = plt.subplots(figsize=(12, 3.8))
        ax.plot(y.index, y.modis_count, marker='o', lw=1.3, color='tab:orange', label='MODIS detections (raw)')
        ax.plot(y.index, y.viirs, marker='o', lw=2, color='black', label='VIIRS detections (actual)')
        ax.plot(y.index, y.harmonized, marker='o', lw=1.6, color='tab:blue', label='Harmonized (VIIRS scale)')
        ax.fill_between(y.index, y.low, y.high, color='tab:blue', alpha=0.2, label='Ensemble band (10–90%)')
        ax.axvline(2012, ls='--', c='gray'); ax.set_yscale('log')
        ax.set_title(f'{a}: yearly fire detections'); ax.set_ylabel('detections (log)'); ax.legend(fontsize=8)
        plt.show()


def plotCalendar(monthly, area, column='calendar'):
    d = monthly[monthly.area == area]
    grid = d.pivot_table(index=d.month.dt.year, columns=d.month.dt.month, values=column, aggfunc='sum')
    fig, ax = plt.subplots(figsize=(9, 0.32 * len(grid) + 1.5))
    im = ax.imshow(np.log1p(grid.to_numpy()), aspect='auto', cmap='inferno')
    ax.set_yticks(range(len(grid)), grid.index); ax.set_xticks(range(12), list('JFMAMJJASOND'))
    ax.axhline(list(grid.index).index(2012) + 0.5 if 2012 in grid.index else -1, color='white', lw=1, ls='--')
    ax.set_title(f'{area}: burning-activity calendar ({column}; dashed line = VIIRS starts)')
    plt.colorbar(im, ax=ax, label='log(1 + detections)'); plt.tight_layout(); plt.show()


def anomalies(monthly, regions, top=5):
    """Months that stand out against the same calendar month in other years (z-score).
    Each region's event year is left out of the baseline, as the dataset plan asks."""
    rows = []
    for a, d in monthly.groupby('area'):
        d = d.copy(); d['mo'], d['yr'] = d.month.dt.month, d.month.dt.year
        r = regions[regions.area == a]
        ev_years = set(range(int(r.focus_start.iloc[0][:4]), int(r.focus_end.iloc[0][:4]) + 1)) if len(r) else set()
        base = d[~d.yr.isin(ev_years)].groupby('mo')['calendar'].agg(['mean', 'std'])
        d = d.join(base, on='mo')
        d['z'] = (d['calendar'] - d['mean']) / d['std'].replace(0, np.nan)
        if len(r):
            d['in_event'] = d.month.between(pd.Timestamp(r.focus_start.iloc[0]), pd.Timestamp(r.focus_end.iloc[0]))
        rows.append(d.nlargest(top, 'z')[['area', 'month', 'calendar', 'source', 'z'] + (['in_event'] if len(r) else [])])
    return pd.concat(rows).round(2)


def saveOutputs(cells, monthly, members, info, report=None):
    """Save the harmonized tables, the model, and a JSON report of the evaluation (read by notebook 5)."""
    os.makedirs(OUTPUT_DIR, exist_ok=True); os.makedirs(MODEL_DIR, exist_ok=True)
    cells.to_parquet(f'{OUTPUT_DIR}/harmonized_cells.parquet', index=False)
    monthly.to_parquet(f'{OUTPUT_DIR}/harmonized_monthly.parquet', index=False)
    joblib.dump({'members': members, 'features': members[0].features, 'grid_km': GRID_KM, **info},
                f'{MODEL_DIR}/harmonization_v2.joblib')
    rep = {'grid_km': GRID_KM, 'years': [START, END], 'viirs_from': VIIRS_FROM,
           'split': {'train_until': TRAIN_UNTIL, 'val_years': list(VAL_YEARS), 'test_from': TEST_FROM},
           'features': list(members[0].features), 'model': OffsetGBM.name, 'members': len(members),
           'generated_at': pd.Timestamp.now(tz='UTC').isoformat(timespec='seconds')}
    for k, v in (report or {}).items():
        if isinstance(v, pd.DataFrame):
            v = json.loads(v.to_json(orient='records', date_format='iso'))
        rep[k] = v
    with open(f'{OUTPUT_DIR}/report.json', 'w') as f:
        json.dump(rep, f, indent=1, default=str)
    print('Saved:', f'{OUTPUT_DIR}/harmonized_cells.parquet', f'{OUTPUT_DIR}/harmonized_monthly.parquet',
          f'{OUTPUT_DIR}/report.json', f'{MODEL_DIR}/harmonization_v2.joblib', sep='\n  ')

## 0. What data do we have?
The backfill needs **MODIS from 2000**. The dataset plan downloads 2013–2024 only, so if this warns you, run in notebook 1:
```python
downloadRegions(years=range(2000, 2013), sensors=('modis',))
```
then notebook 2's *all regions* loop again.

In [ ]:
AREAS = [a for a in availableAreas()]      # or a list, e.g. ['Egypt_Nile_Delta', 'Portugal_Centro']
regions = loadRegions()
coverage(AREAS)

## 1. Build the cell × month table
Every 10 km cell that ever had a detection (either sensor), every month 2000–2024. Features are MODIS only:

| Feature | Meaning |
|---|---|
| `modis_count`, `terra_count`, `aqua_count` | MODIS detections in the cell that month |
| `frp_sum`, `frp_max`, `frp_mean` | Fire power |
| `pixel_area_mean` | MODIS pixel size: edge-of-swath pixels hide small fires |
| `day_share` | Share of daytime detections |
| `nb_count` | MODIS detections in the 8 neighbouring cells |
| `cell_mean`, `cell_month_mean` | The cell's long-term MODIS activity, overall and for that calendar month |
| `month_sin`, `month_cos` | Season |
| `region_*` | Which region (fire type) |

Target `viirs_count` exists from 2013; earlier months are what the model fills in.

In [ ]:
df = buildCellMonth(AREAS)
region_cols = addRegionColumns(df, AREAS)
FEATURES = BASE_FEATURES + region_cols
df.head()

## 2. Split by year
Train 2013–2020, validate 2021, test 2022–2024. Never random: neighbouring cells and months of one fire must not land in both train and test.

In [ ]:
train, val, test = splitByYear(df)

## 3. Compare models (validation year)
- **Average ratio:** per region, VIIRS total / MODIS total, times each cell's MODIS count. Cells where MODIS saw nothing get 0.
- **Plain gradient boosting:** learns VIIRS counts from the MODIS features. Fits cells well but can't predict beyond what it saw, so it underestimates the biggest fire months.
- **Ratio × learned correction (main):** starts from the ratio (and, for cells where MODIS saw nothing, the region's typical VIIRS count there) and learns how much to scale it. Big fires keep scaling with MODIS; the blind spot is learned.

Scores: **Poisson deviance (cell)** lower is better · **Total bias %** · **Monthly WAPE %** error on monthly region totals, what the calendar shows · **Monthly corr**.

In [ ]:
results, fitted = compareModels(train, val, FEATURES)
results.round(3)

## 4. Final test, once
All three models refit on 2013–2021 and scored once on 2022–2024; then the main model per region, with monthly totals plotted against real VIIRS.

In [ ]:
gbm, test_overall, test_by_region = finalTest(train, val, test, FEATURES)
display(test_overall.round(3))
test_by_region.round(3)

## 5. Does it work in a region it never saw?
Trains on the other regions without the region columns. With very different fire types this is hard; it improves as more regions are added.

In [ ]:
observed = df[df[TARGET].notna()]
leaveOneRegionOut(observed, BASE_FEATURES).round(3)

## 6. Final ensemble and backfill 2000–2024
5 copies of the main model, each on a bootstrap resample of all observed years (2013–2024). Their mean is the harmonized value; their 10–90% spread is the band.

**Note:** the band shows model uncertainty only, not the year-to-year noise in fire counts, so it is narrower than the true error. The test WAPE above is the honest error size.

In [ ]:
members = trainEnsemble(observed, FEATURES)
cells, monthly = harmonize(df, members)
monthly.head()

## 7. The 2012 check (headline result)
Compares 2009–2011 with 2013–2015 in each region:
- **raw jump:** joining MODIS then VIIRS, the fake step at the switch
- **real change:** MODIS on both sides, the true change in fire activity
- **harmonized jump:** should be close to the real change

In [ ]:
jumpCheck(monthly)

In [ ]:
plotYearly(monthly)

## 8. Burning-activity calendars
Real VIIRS from 2013, the model before that.

In [ ]:
for a in AREAS:
    plotCalendar(monthly, a)

## 9. Do the known events stand out?
Top months per region by z-score against the same calendar month in other years, leaving the region's event year out of the baseline. `in_event` = inside the focus period from the dataset plan.

In [ ]:
anomalies(monthly, regions)

## 10. Save
Tables, model and `report.json` (all the scores above) for notebook 5, which exports everything for the web app.

In [ ]:
loro = leaveOneRegionOut(observed, BASE_FEATURES)
saveOutputs(cells, monthly, members, {'areas': AREAS},
            report={'validation': results, 'test_overall': test_overall, 'test_by_region': test_by_region,
                    'leave_one_region_out': loro, 'jump_check': jumpCheck(monthly),
                    'anomalies': anomalies(monthly, regions, top=8)})